# Future Planning – Scan-Worker (Colab / T4)

Dünner Starter für den GPU-Scan-Worker. **Ablauf:**

1. Repo `FP_APP` klonen (flach).
2. Scan-Worker + GPU-Stack installieren (eine Setup-Zelle).
3. Gradio-Worker mit `share=True` starten → Scan-Bundle (`video.*` + `poses.json`) hochladen.
4. Ergebnis: `scene.ply` (z-up, metrisch) + `layout.txt` (SpatialLM) für den Server-Adapter.

Der Geometrie-Kern läuft CPU-only; hier auf Colab kommen Depth Anything V2 Small + SpatialLM dazu.
Deploy-Idee (Worker + Space, Zeiger v0) steht im Brain: **POC-Demo-Architektur-HF**.

In [ ]:
# Zelle 2 – Repo flach klonen (privates Repo → GITHUB_PAT aus Colab-Secrets)
from google.colab import userdata
import os, subprocess

PAT = userdata.get("GITHUB_PAT")  # in Colab: 🔑 Secrets → GITHUB_PAT hinterlegen
URL = f"https://{PAT}@github.com/Bryan-HSLU/FP_APP"
if not os.path.isdir("FP_APP"):
    subprocess.run(["git", "clone", "--depth", "1", URL, "FP_APP"], check=True)
os.chdir("FP_APP")
print("cwd:", os.getcwd())

In [ ]:
# Zelle 3 – EINE Setup-Zelle: Worker-Kern + SpatialLM-1.1-Stack (GPU/T4)
import os, subprocess

# 1) Worker-Kern + permissive Colab-Bausteine (Depth Anything V2 Small = Apache):
#    torch/transformers/opencv/open3d – für Tiefe, Fusion, PLY, Outlier-Filter.
get_ipython().system("pip -q install -e services/scan-worker[worker] opencv-python-headless open3d torch transformers")

# 2) SpatialLM 1.1 (CC-BY-NC – NUR Colab, NIE feste Dependency! CLAUDE.md §4).
#    Offizieller Install laut SpatialLM-README: Env mit torch 2.4.1 / CUDA 12.4,
#    Poetry, danach `poe install-sonata`. SpatialLM 1.1 nutzt das Sonata-Backbone
#    (+ flash-attn) – NICHT TorchSparse (das war nur SpatialLM 1.0).
#    → Der flash-attn-Build ist der Zeitfresser. Wir cachen das Wheel auf Google
#      Drive: nur die ERSTE Session kompiliert, danach Sekunden statt Minuten.
SPATIALLM_DIR = "/content/SpatialLM"
FP_APP_DIR = os.getcwd()
if not os.path.isdir(SPATIALLM_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/manycore-research/SpatialLM", SPATIALLM_DIR],
        check=True,
    )

from google.colab import drive
drive.mount("/content/drive")
WHEEL_CACHE = "/content/drive/MyDrive/fp_wheels"
os.makedirs(WHEEL_CACHE, exist_ok=True)

# Poetry in die Colab-Umgebung installieren (kein eigenes venv → nutzt Colab-Python):
os.chdir(SPATIALLM_DIR)
get_ipython().system("pip -q install poetry && poetry config virtualenvs.create false --local")
get_ipython().system("poetry install")

# flash-attn: erst aus dem Drive-Cache; fehlt es dort, einmal bauen und sichern.
aus_cache = subprocess.run(
    ["pip", "install", "--no-index", f"--find-links={WHEEL_CACHE}", "flash-attn"]
).returncode == 0
if not aus_cache:
    get_ipython().system("poe install-sonata")            # baut flash-attn (langsam, einmalig)
    get_ipython().system(f"pip wheel flash-attn -w {WHEEL_CACHE}")  # Wheel für nächste Session sichern

os.chdir(FP_APP_DIR)

# 3) Worker sagen, wo SpatialLM-Repo + Modell liegen (liest worker._lauf_spatiallm):
os.environ["FP_SPATIALLM_DIR"] = SPATIALLM_DIR
os.environ["FP_SPATIALLM_MODELL"] = "manycore-research/SpatialLM1.1-Qwen-0.5B"

# ⚠️ GATE (Fahrplan Schritt 5): Colab bringt oft eine NEUERE torch-Version mit
#    als die von SpatialLM erwarteten 2.4.1 / CUDA 12.4. Falls `poetry install`
#    oder der flash-attn-Build hier bricht, die funktionierenden Pins beim ersten
#    echten R1-Lauf ermitteln und GENAU HIER eintragen (dann ist der Gate durch).


In [ ]:
# Zelle 4 – Worker starten (öffentliche share-URL)
from fp_scan_worker.worker import erstelle_app

app = erstelle_app()
app.launch(share=True)
# → Zeiger v0: die ausgegebene *.gradio.live-URL MANUELL als FP_SCAN_WORKER_URL
#   im Hugging-Face-Space hinterlegen. Gist-Automation folgt später.

## Colab-Stolperfallen

- **Grosse Videos** nicht direkt hochladen – über Google Drive einbinden (`drive.mount`), sonst reisst der Upload ab.
- **Session stirbt** (Timeout / Neustart) → Laufzeit ist weg: Setup-Zelle (Zelle 3) erneut ausführen. Mit Drive-Wheel-Cache bleibt das schnell.
- **share-URL wechselt** bei jedem `launch` → nach jedem Neustart `FP_SCAN_WORKER_URL` im Space aktualisieren (Zeiger v0).